In [0]:
%pip install PyMuPDF azure-ai-formrecognizer python-docx

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from pyspark.sql.functions import col, length
import fitz

from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential

In [0]:
print("JOB STARTED")

endpoint="endpoint"
key="key"

print("Initializing Azure Form Reciognizer Client")

client = DocumentAnalysisClient(endpoint=endpoint, credential=AzureKeyCredential(key))

print("Azure client initialized")
print("Reading files from DBFS")

pdf_df=spark.read.format("binaryFile").option("recursiveFileLookup","true").load("dbfs:/FileStore/noc_pm_pdfs").select("path", "content")

print("PDF read completed")

print("Loading already processed files")

try:
    processed_df=(
        spark.table("noc_pm_documents_backup")
        .filter(col("final_text").isNotNull() & (length(col("final_text")) > 0))
        .select("path"))
    
    processed_paths=set(row["path"] for row in processed_df.collect())

    print(f"Already processed documents found: {len(processed_paths)}")

except Exception:
    processed_paths=set()

    print("Table noc_pm_documents_backup does not exist")

def extract_text_from_pdf(binary, path):

    print(f"[FITZ] Extracting Text")
    
    try:
        doc=fitz.open(stream=binary, filetype="pdf")
        text= "\n".join(page.get_text() for page in doc)
        print(f"[FITZ] Success ({len(text)}chars)")
        return text
    
    except Exception as e:
        print(f"[FITZ ERROR] {path} -> {e}")
        return ""

def ocr_pdf(binary, path):
    print(f"[OCR] Extracting Text")
    try:
        poller = client.begin_analyze_document("prebuilt-read", document=binary)
        result = poller.result()
        text="\n".join(
            line.content
            for page in result.pages
            for line in page.lines
        )
        print(f"[OCR] Success ({len(text)}chars)")
        return text
    except Exception as e:
        print(f"[OCR ERROR] {path} -> {e}")
        return ""


schema=StructType([
    StructField("path", StringType(), True),
    StructField("final_text", StringType(), True),
    StructField("is_scanned", BooleanType(), True)
])

rows=pdf_df.toLocalIterator()

for idx, row in enumerate(rows, start=1):
    path=row["path"]
    binary=row["content"]

    if path in processed_paths:
        print(f"Skipping already processed file: {path})")
        continue
    try:
        text= extract_text_from_pdf(binary, path)
        is_scanned=len(text)<200
        print(f"[CHECK] Scanned PDF: {is_scanned}")

        if is_scanned:
            final_text=ocr_pdf(binary, path)
        else:
            print("[SKIP OCR] Using PDF text")
            final_text=text

        result_df=spark.createDataFrame([(path, final_text, is_scanned)], schema)
        result_df.write.format("delta").mode("append").saveAsTable("noc_pm_documents_backup")

        print("[SUCCESS] File processed")
    except Exception as e:
        print(f"[FATAL ERROR] {path} -> {e}")

print("JOB FINISHED")

In [0]:
display(
    spark.table("noc_pm_documents")
         .select("path", "is_scanned", "final_text")
         .orderBy("path")
)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from pyspark.sql.functions import col, length
import fitz
from docx import Document
import os

#===============================================================================
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential

print("JOB STARTED")

endpoint="endpoint"
key="key"

print("Initializing Azure Form Recognizer Client")
client = DocumentAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)
print("Azure client initialized")

#===============================================================================

print("Reading files from DBFS")
pdf_df = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load("dbfs:/FileStore/noc_pm_pdfs")
    .select("path", "content")
)
print("PDF read completed")
#===============================================================================
print("Loading already processed files")
try:
    processed_df = (
        spark.table("noc_pm_documents_backup")
        .filter(col("final_text").isNotNull() & (length(col("final_text")) > 0))
        .select("path")
    )
    processed_paths = set(row["path"] for row in processed_df.collect())
    print(f"Already processed documents found: {len(processed_paths)}")
except Exception:
    processed_paths = set()
    print("Table noc_pm_documents_backup does not exist")

print("loading Unprocessed files")

#===============================================================================
unprocessed_df = pdf_df.join(
    processed_df,
    on="path",
    how="left_anti"
)
#===============================================================================



def ocr_pdf(binary, path):
    print("[OCR] Extracting Text")
    try:
        poller = client.begin_analyze_document(
            "prebuilt-read",
            document=binary
        )
        result = poller.result()
        text = "\n".join(
            line.content
            for page in result.pages
            for line in page.lines
        )
        print(f"[OCR] Success ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"[OCR ERROR] {path} -> {e}")
        return ""

#===============================================================================

def extract_text_from_pdf(binary, path):
    print("[FITZ] Extracting Text")
    try:
        doc = fitz.open(stream=binary, filetype="pdf")
        text = "\n".join(page.get_text() for page in doc)
        print(f"[FITZ] Success ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"[FITZ ERROR] {path} -> {e}")
        return ""

def extract_text_from_plain(binary, path):
    print("[PLAIN] Extracting Text")
    try:
        text = binary.decode("utf-8", errors="ignore")
        print(f"[PLAIN] Success ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"[PLAIN ERROR] {path} -> {e}")
        return ""

def extract_text_from_docx(binary, path):
    print("[DOCX] Extracting Text")
    try:
        tmp_path = f"/tmp/{os.path.basename(path)}"
        with open(tmp_path, "wb") as f:
            f.write(binary)

        doc = Document(tmp_path)
        text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
        print(f"[DOCX] Success ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"[DOCX ERROR] {path} -> {e}")
        return ""

def extract_text(binary, path):
    ext = os.path.splitext(path.lower())[1]

    if ext == ".pdf":
        text = extract_text_from_pdf(binary, path)

    elif ext == ".docx":
        text = extract_text_from_docx(binary, path)

    elif ext == ".doc":
        print("[DOC] Legacy .doc detected → OCR only")
        return ocr_pdf(binary, path)

    elif ext in {".txt", ".csv", ".md", ".rtf"}:
        text = extract_text_from_plain(binary, path)

    else:
        print(f"[INFO] Unsupported format {ext}, using OCR")
        text = ""

    return text

#===============================================================================

schema = StructType([
    StructField("path", StringType(), True),
    StructField("final_text", StringType(), True),
    StructField("is_scanned", BooleanType(), True)
])

#===============================================================================
rows = unprocessed_df.toLocalIterator()

for idx, row in enumerate(rows, start=1):
    path = row["path"]
    binary = row["content"]

    # if path in processed_paths:
    #     print(f"Skipping already processed file: {path}")
    #     continue

    try:
        text = extract_text(binary, path)
        is_scanned = len(text) < 200
        print(f"[CHECK] Scanned PDF: {is_scanned}")

        if is_scanned:
            final_text = ocr_pdf(binary, path)
        else:
            print("[SKIP OCR] Using PDF text")
            final_text = text

        result_df = spark.createDataFrame(
            [(path, final_text, is_scanned)],
            schema
        )

        result_df.write.format("delta").mode("append").saveAsTable("noc_pm_documents_backup")

        print("[SUCCESS] File processed")

    except Exception as e:
        print(f"[FATAL ERROR] {path} -> {e}")

print("JOB FINISHED")

df = spark.table("noc_pm_documents_backup")
print("Total records saved:", df.count())
df.select("path", "is_scanned", "final_text").orderBy("path").show(10)


In [0]:
df = spark.table("noc_pm_documents")
print("Total records saved:", df.count())
df.select("path", "is_scanned", "final_text").orderBy("path").show(10)

In [0]:
from pyspark.sql.functions import col
display(

    spark.table("noc_pm_documents")
         .filter(col("path") == "dbfs:/FileStore/noc_pm_pdfs/0000008498.DOCX")
         .select("path", "is_scanned", "final_text")
)

In [0]:
from pyspark.sql.functions import lower, col, length
display(
    
    spark.table("noc_pm_documents")

         .filter(lower(col("path")).endswith(".docx"))
         .withColumn("text_len", length(col("final_text")))
         .select("path", "is_scanned", "text_len", "final_text")
         .orderBy(col("text_len").asc())
)

In [0]:
from pyspark.sql.functions import lower, col, length
display(
    
    spark.table("noc_pm_documents")

         .filter(lower(col("path")).endswith(".pdf"))
         .withColumn("text_len", length(col("final_text")))
         .select("path", "is_scanned", "text_len", "final_text")
         .orderBy(col("text_len").asc())
)